# Google Colab Example for eDensGPU_MP2

This notebook runs an example MP2 correlation energy density evaluation on Google Colab. It showcases the GPU-based implementation that uses the auxiliary basis expansion. 

## Table of contents 
- [Installation](#1-installation)
- [Molecule Setup](#2-molecule-setup)
- [Run `ec_mp2_gen`](#3-run-edensgpu_mp2)

## 1) Installation
For the Google Colab environment setup, first select a suitable runtime:

**Runtime → Change runtime type → Runtime Version → 2026.07**, with
any **NVIDIA GPU**. 

This runtime allows **Python 3.12** (see
[Colab's runtime list](https://research.google.com/colaboratory/runtime-version-faq.html)).

Installation uses the latest commit on the public GitHub `main` branch. Set `REVISION` to a release tag to install a fixed version instead. 

The following `pip` command installs **CuPy, GPU4PySCF, cuTENSOR** and the pinned numerical libraries. Run the installation cell below first, before importing scientific packages. Continue directly to section 2 when installation succeeds.

In [ ]:
import importlib
import json
import importlib.metadata as metadata
import os
import subprocess
import sys

REPOSITORY = "https://github.com/vuckovic-lab/eDensGPU_MP2.git"
REVISION = "main"  # Replace with a release tag to install a specific version; "main" installs the latest commit. 
 
if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Select Colab runtime 2026.07 (Python 3.12) with an NVIDIA GPU.")
print("Checking for an NVIDIA GPU ...")
subprocess.run(["nvidia-smi"], check=True)
print(f"Installing eDensGPU_MP2 from {REVISION} ...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-input",
     f"git+{REPOSITORY}@{REVISION}"],
    check=True,
    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
)
importlib.invalidate_caches()
print("Installed package versions:")
for label, distribution in (
    ("NumPy", "numpy"),
    ("GPU4PySCF", "gpu4pyscf-cuda12x"),
    ("CuPy", "cupy-cuda12x"),
    ("eDensGPU_MP2", "eDensGPU_MP2"),
):
    print(f"  {label}: {metadata.version(distribution)}")

# pip records the installed Git commit; match tags to that exact commit.
# A checkout from main may have no release tag.
try:
    direct_url = metadata.distribution("eDensGPU_MP2").read_text("direct_url.json")
    provenance = json.loads(direct_url or "{}")
    commit = provenance.get("vcs_info", {}).get("commit_id")
    if commit:
        print(f"  Installed Git commit: {commit}")
        remote_tags = subprocess.run(
            ["git", "ls-remote", "--tags", REPOSITORY],
            check=True, capture_output=True, text=True, timeout=30,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        )
        tags = {}
        for line in remote_tags.stdout.splitlines():
            sha, ref = line.split()
            name = ref.removeprefix("refs/tags/").removesuffix("^{}")
            # Annotated tags use the peeled commit, not the tag object's SHA.
            if ref.endswith("^{}") or name not in tags:
                tags[name] = sha
        matching = sorted(name for name, sha in tags.items() if sha == commit)
        print("  Git tag: " + (", ".join(matching) if matching else "no tag at installed commit"))
    else:
        print("  Git tag: unavailable (no installed Git commit metadata)")
except (OSError, subprocess.SubprocessError, ValueError) as exc:
    print(f"  Git tag lookup unavailable: {exc}")

print("Installation complete. Continue with section 2; restart the session if already-loaded packages changed.")


## 2) Molecule setup

Choose your molecular system and its parameters for the PySCF evaluation. 
Note that PySCF uses `spin=2S`. Set `scf_type` to `rhf` for a closed-shell system or `uhf` for an open-shell system. 



In [ ]:
import time
import cupy as cp
from pyscf import gto
from gpu4pyscf import dft

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("Select an NVIDIA GPU runtime.")

###==== Define molecule and basis set ====###
struc = """
O   0.000000   0.000000   0.000000
H   0.000000  -0.757000   0.587000
H   0.000000   0.757000   0.587000
"""
charge = 0
spin = 0
basis = "def2qzvppd"
scf_type = "rhf"       # "rhf" or "uhf"
xc_fun = 'hf'


###==== Run SCF calculation ==###
mol = gto.M(atom=struc, basis=basis, charge=charge, spin=spin, cart=False)
if scf_type == "rhf":
    mf = dft.RKS(mol)
    mf.xc = xc_fun
elif scf_type == "uhf":
    mf = dft.UKS(mol)
    mf.xc = xc_fun
cp.cuda.runtime.deviceSynchronize()
scf_started = time.perf_counter()
mf.kernel()
cp.cuda.runtime.deviceSynchronize()
scf_seconds = time.perf_counter() - scf_started
print("HF energy / Hartree:", repr(float(mf.e_tot)))
print(f"SCF elapsed: {scf_seconds:.3f} s")


## 3) Run `eDensGPU_MP2`
MP2 correlation energy density generation `ec_mp2_gen()` uses regularization `kappa=2.0` as the default. Choose the grid level and fitting basis independently of the orbital basis.

The returned `ec_out` class object contains `tot` (total) and `os` (opposite spin) correlation energies in Hartree, `dens_tot` and `dens_os` energy densities per particle for total and opposite-spin channels, and `rho` as the electronic density, all evaluated on the generated grid `coords` with integration weights `weights` from `make_grid()`.


In [ ]:
import time
import cupy as cp
from eDens_mp2 import ec_mp2_gen, make_grid

###==== eDensGPU_MP2 parameters ====###
grid_level = 3
kappa = 2.0
auxbasis = basis + "ri"
coords, weights = make_grid(mol, level=grid_level)

###==== Run eDensGPU_MP2 calculation ====###
cp.cuda.runtime.deviceSynchronize()
mp2_started = time.perf_counter()
ec_out = ec_mp2_gen(mf, mol, coords, weights, kappa=kappa, auxbasis=auxbasis, verbose=True)
cp.cuda.runtime.deviceSynchronize()
mp2_seconds = time.perf_counter() - mp2_started
print(f"Generation elapsed: {mp2_seconds:.3f} s")

print("Correlation energies / Hartree:")
print("Total:", repr(ec_out.tot))
print("OS:   ", repr(ec_out.os))
